In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import os

from utils import set_working_directories, get_wake_data, load_from_mat_file


data_folder = set_working_directories('wakes')[0]
case = 'triangle_re_100'

get_wake_data(data_folder, case=case)



In [ ]:
mat = load_from_mat_file(f'{data_folder}/{case}.mat')
data = np.array([mat['ux'], mat['uy']]) 
data[np.isnan(data)] = 0.



In [ ]:
Nu, Nt, Nx, Ny = data.shape
grid_shape = (Nu, Nx, Ny)

data = data.transpose(0, 2, 3, 1)  # We want data dimensions as (Nu, Nx, Ny, Nt) 

N_test = int(.6 * Nt)
X_test = data[..., -N_test:].copy()
X_train = data[..., :-N_test].copy()

In [ ]:
from modulo_vki.modulo import ModuloVKI as Modulo


def run_snapshot_POD(Q):
    """ 
        Snapshot POD decomposition using MODULO
        X(x, t) = Φ(t) Σ Ψ(x), where X = Q - Q_mean
        returns Φ, Ψ, Σ
        
    """

    Q = Q.reshape((-1, Q.shape[-1]))  
    Q_mean = np.mean(Q, axis=-1, keepdims=True)

    m = Modulo(data=Q - Q_mean, n_Modes=N_modes)  
    
    return m.POD() 


In [ ]:
# help(Modulo.POD)

In [ ]:
N_modes = 10
Psi, Phi, Sigma = run_snapshot_POD(Q=X_train)

In [ ]:

def plot_POD_modes(cmap='viridis'):

    POD_basis = np.reshape(Psi, shape=(*grid_shape, N_modes))

    x1 = np.arange(grid_shape[1])
    x2 = np.arange(grid_shape[2])

    X1, X2 = np.meshgrid(x1, x2, indexing='ij')

    n_col = min(N_modes, 5)
    n_row = int((N_modes + 1) // n_col)

    for jj, data, ttl in zip(range(POD_basis.shape[0]), POD_basis, ['$u_x$', '$u_y$']):
        fig1 = plt.figure(figsize=(3.5 * n_col + 1., 2.5 * n_row), layout='constrained')
        axs = fig1.subplots(nrows=n_row, ncols=n_col, sharex=True, sharey=True)

        for kk, ax in zip(range(N_modes), axs.ravel()):
            im0 = ax.pcolormesh(X1, X2, data[...,kk], rasterized=True)

            if kk >= N_modes - n_col:
                ax.set(xlabel='$y$')
            if kk % n_col == 0:
                ax.set(ylabel='$x$')
            if (kk + 1) % n_col == 0:
                fig1.colorbar(mappable=im0, ax=ax, label=ttl)

            ax.set(title=f'mode {kk+1}')
            


def plot_spectrum(Q, max_mode=None):
    """
    The energy of each POD mode is given by λ_i / 2, where λ_i = sigma_i^2
    """

    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
    Lambda = Sigma ** 2

    axs[0].bar(np.arange(N_modes), Lambda / Lambda[0], color='C4')
    axs[0].set(xlabel='Mode number $j$', title='$\\lambda_j / \\lambda_0$', xlim=[-1, max(N_modes, 10)])
    axs[1].plot(np.arange(N_modes), np.cumsum(Lambda) / sum(Lambda), color='C4',
                 label='$\\dfrac{\\sum_{j}{\,\\lambda_j}}{ \\sum_{k=0}^{' + f'{N_modes}' + '}\\lambda_k}$')

    # Compute the total kinetic energy of the system to checjh the POD energy reconstruction
    Q -= np.mean(Q, axis=-1, keepdims=True)
    TKE = 0.5 * np.sum((np.mean(Q[0] ** 2, axis=2) + np.mean(Q[1] ** 2, axis=2)))

    POD_energy = Lambda / (2 * Phi.shape[0])
    axs[1].plot(np.arange(N_modes), np.cumsum(POD_energy) / TKE, dashes=[10, 2], color='k', 
                label='$\\dfrac{\\sum_{j}{\,\\lambda_j}}{\mathrm{TKE}}$')
    axs[1].grid(visible=True, linestyle='--', alpha=0.5)
    axs[1].set(xlabel='Mode number $j$', title='Cumulative energy', xlim=[0, max_mode])
    axs[1].legend(ncol=2)
    axs[0].set(ylim=[0, None],  xlim=[-1, max_mode])

In [ ]:
plot_spectrum(Q=X_train)

In [ ]:
plot_POD_modes()

# We can do the same with the POD class

In [ ]:
from tools_ML.POD import POD

print(X_train.shape)

pod_object = POD(X=X_train.transpose(0, 2, 1, 3), 
                 domain=[-2, 2, 0, 12])
                 

In [ ]:
POD.plot_POD_modes(pod_object, cmap='viridis')

In [ ]:
help(POD)